Лабораторная работа
Подготовка обучающей и тестовой выборки, кросс-валидация и подбор гиперпараметров на примере метода ближайших соседей.

Цель лабораторной работы: изучение способов подготовки выборки и подбора гиперпараметров на примере метода ближайших соседей.
Требования к отчету:

Отчет по лабораторной работе должен содержать:

    титульный лист;
    описание задания;
    текст программы;
    экранные формы с примерами выполнения программы.

Задание:

    Выберите набор данных (датасет) для решения задачи классификации или регрессии.
    В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.
    С использованием метода train_test_split разделите выборку на обучающую и тестовую.
    Обучите модель ближайших соседей для произвольно заданного гиперпараметра K. Оцените качество модели с помощью подходящих для задачи метрик.
    Произведите подбор гиперпараметра K с использованием GridSearchCV и RandomizedSearchCV и кросс-валидации, оцените качество оптимальной модели. Используйте не менее двух стратегий кросс-валидации.
    Сравните метрики качества исходной и оптимальной моделей.


In [1]:
import pandas as pd  # библиотека для работы с таблицами (DataFrame)
import numpy as np  # библиотека для численных операций и массивов
from sklearn.preprocessing import StandardScaler  # масштабирование признаков
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, cross_val_score, KFold, StratifiedKFold  # инструменты для разделения и валидации
from sklearn.neighbors import KNeighborsClassifier  # алгоритм K-ближайших соседей
from sklearn.metrics import classification_report  # метрики классификации
from sklearn.model_selection import StratifiedShuffleSplit  # стратифицированное случайное разбиение

In [2]:
# Устанавливаем зерно генератора случайных чисел для воспроизводимости
np.random.seed(42)

# Загружаем CSV-файл с данными и выбираем 3% случайных строк для быстрой отладки
df = pd.read_csv("./coffee_shop_sales.csv").sample(frac=0.03, random_state=42)

In [3]:
# 1. Создаём бинарную целевую переменную: 1, если transaction_qty выше медианы, иначе 0
threshold = df["transaction_qty"].median()  # медианное значение количества транзакций
df["High_Transaction"] = (df["transaction_qty"] > threshold).astype(int)  # создаём новую колонку на основе сравнения

# 2. Проверяем, сбалансированы ли классы
print("Баланс классов:\n", df["High_Transaction"].value_counts(normalize=True))  # выводим процентное распределение классов

Баланс классов:
 High_Transaction
0    0.586631
1    0.413369
Name: proportion, dtype: float64


In [4]:
# 3. Выбираем признаки для обучения (X) и целевую переменную (y)
# Убираем transaction_qty из X, чтобы не допустить утечки информации
X = df[['store_id', 'product_id', 'unit_price', 'Total_Bill', 'Hour', 'Month', 'Day of Week']]
y = df['High_Transaction']

# 4. Масштабируем числовые признаки с помощью StandardScaler (среднее = 0, std = 1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # получаем масштабированные признаки в виде numpy-массива

# 5. Разбиваем выборку на train/test, сохраняя пропорции классов с помощью StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)  # 1 разбиение, 20% — тест
for train_index, test_index in sss.split(X_scaled, y):  # получаем индексы для обучающей и тестовой выборок
    X_train, X_test = X_scaled[train_index], X_scaled[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# 6. Создаём и обучаем модель K-ближайших соседей с k=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)  # обучаем на тренировочных данных
y_pred = knn.predict(X_test)  # предсказываем классы на тесте

# 7. Выводим метрики классификации для модели с K=5
print("Метрики качества для K=5:")
print(classification_report(y_test, y_pred))  # precision, recall, f1-score

# 8. Подбор оптимального значения K через GridSearchCV (полный перебор)
param_dist = {'n_neighbors': np.arange(1, 21)}  # будем перебирать значения от 1 до 20
grid_search = GridSearchCV(KNeighborsClassifier(), param_dist, cv=5, scoring='accuracy')  # 5-кратная кросс-валидация
grid_search.fit(X_train, y_train)  # подбираем лучший k
print("Лучшее значение K по GridSearchCV:", grid_search.best_params_)

# 9. Подбор оптимального значения K через RandomizedSearchCV (случайный перебор)
random_search = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_distributions=param_dist,
    n_iter=10,  # случайно пробуем 10 значений
    cv=5,
    scoring='accuracy',
    random_state=42
)
random_search.fit(X_train, y_train)
best_k = random_search.best_params_['n_neighbors']  # сохраняем лучшее найденное значение K
print(f"Лучшее значение K по RandomizedSearchCV: {best_k}")

# 10. Кросс-валидация с использованием обычного KFold (без сохранения баланса классов)
kf = KFold(n_splits=5, shuffle=True, random_state=42)  # делим на 5 фолдов с перемешиванием
kf_scores = cross_val_score(
    KNeighborsClassifier(n_neighbors=grid_search.best_params_['n_neighbors']),
    X_train, y_train, cv=kf, scoring='accuracy'
)
print("Средняя точность по KFold:", np.mean(kf_scores))

# 11. Кросс-валидация с StratifiedKFold (с сохранением баланса классов)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_scores = cross_val_score(
    KNeighborsClassifier(n_neighbors=grid_search.best_params_['n_neighbors']),
    X_train, y_train, cv=skf, scoring='accuracy'
)
print("Средняя точность по StratifiedKFold:", np.mean(skf_scores))

# 12. Обучаем финальную модель с наилучшим найденным K
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train, y_train)
y_best_pred = best_knn.predict(X_test)  # делаем предсказания на тестовой выборке

# 13. Выводим финальные метрики для лучшей модели
print("Метрики качества для оптимального K:")
print(classification_report(y_test, y_best_pred))


Метрики качества для K=5:
              precision    recall  f1-score   support

           0       0.79      0.83      0.81       525
           1       0.74      0.68      0.71       370

    accuracy                           0.77       895
   macro avg       0.76      0.76      0.76       895
weighted avg       0.77      0.77      0.77       895

Лучшее значение K по GridSearchCV: {'n_neighbors': np.int64(17)}
Лучшее значение K по RandomizedSearchCV: 17
Средняя точность по KFold: 0.7593671133335939
Средняя точность по StratifiedKFold: 0.7705367816541002
Метрики качества для оптимального K:
              precision    recall  f1-score   support

           0       0.78      0.88      0.83       525
           1       0.79      0.66      0.72       370

    accuracy                           0.79       895
   macro avg       0.79      0.77      0.77       895
weighted avg       0.79      0.79      0.78       895

